# Season database explorer

**This notebook does not run the pipeline.** Ingestion and feature computation are
CLI commands so they can be scheduled, budgeted, and logged — see the README.

This is for looking at a season database once it exists.


In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd

from bsetl.transform.metadata import compute_season_metadata

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Point this at any season database under data/seasons/
DB = Path('../data/seasons/season42/season42_combined_skill_ns.db')
assert DB.exists(), f'not found: {DB}'

con = sqlite3.connect(f'file:{DB}?mode=ro', uri=True)
print(DB.name, f'({DB.stat().st_size / 1e6:.0f} MB)')


## Shape and coverage


In [ ]:
summary = pd.read_sql_query('''
    SELECT COUNT(*)                AS matches,
           MIN(battle_time)        AS first_set,
           MAX(battle_time)        AS last_set,
           COUNT(DISTINCT mode)    AS modes,
           COUNT(DISTINCT map)     AS maps,
           ROUND(AVG(avg_elo), 2)  AS mean_avg_elo
    FROM matches
''', con)
summary.T.rename(columns={0: 'value'})


## A few rows


In [ ]:
pd.read_sql_query('''
    SELECT battle_time, mode, map, record, star_brawler, star_elo, avg_elo,
           t1_b0_name, t1_b1_name, t1_b2_name,
           t2_b0_name, t2_b1_name, t2_b2_name
    FROM matches
    ORDER BY battle_time DESC
    LIMIT 10
''', con)


## Skill feature coverage

`skill_ns` is the time-local ECDF normalization of `avg_elo`; `skill_ns_ok` flags
whether the row's time bin had enough samples to trust it. Absent if the feature
has not been computed for this database.


In [ ]:
cols = {r[1] for r in con.execute('PRAGMA table_info(matches)')}

if 'skill_ns' in cols:
    display(pd.read_sql_query('''
        SELECT skill_ns_ok,
               COUNT(*)                 AS rows,
               ROUND(AVG(skill_ns), 4)  AS mean_skill_ns,
               ROUND(MIN(skill_ns), 3)  AS min_skill_ns,
               ROUND(MAX(skill_ns), 3)  AS max_skill_ns
        FROM matches
        GROUP BY skill_ns_ok
    ''', con))
else:
    print('skill_ns not present — run: bsetl-skill-features --clean-db-path', DB)


## Mode and map distribution


In [ ]:
by_mode = pd.read_sql_query('''
    SELECT mode, COUNT(*) AS matches
    FROM matches GROUP BY mode ORDER BY matches DESC
''', con)
by_mode['share'] = (by_mode.matches / by_mode.matches.sum()).round(3)
by_mode


## Brawler pick counts

Each set contributes six picks, one per slot across both teams.


In [ ]:
slots = [f't{t}_b{b}_name' for t in (1, 2) for b in range(3)]
union = ' UNION ALL '.join(
    f'SELECT {c} AS name FROM matches WHERE {c} IS NOT NULL' for c in slots
)
picks = pd.read_sql_query(
    f'SELECT name, COUNT(*) AS picks FROM ({union}) GROUP BY name ORDER BY picks DESC',
    con,
)
picks['pick_rate'] = (picks.picks / (len(picks) and picks.picks.sum())).round(4)
print(f'{len(picks)} distinct brawlers')
picks.head(20)


## Full metadata sidecar

Same computation the publish step writes alongside each released season.
Scans every row, so it takes a moment on a multi-million-row database.


In [ ]:
meta = compute_season_metadata(str(DB))
{k: v for k, v in meta.items() if k not in ('unique_brawlers', 'maps', 'brawler_usage_top')}


In [ ]:
con.close()
